# Paralelne grane cjevovoda — raspodjela protoka

**Poglavlje U13: Cjevovodi**

Ovaj interaktivni prikaz nadopunjuje rad s paralelnim spojem cijevi. Mijenjanjem geometrije dvije grane i ukupnog protoka prati se kako se taj protok dijeli između grana uz uvjet jednakog pada ukupne energije.

## Cilj

U paralelnom spoju dvije ili više cijevi između istih čvorova, raspodjela protoka po granama nije slobodna nego je određena uvjetom da svaka grana 'plati' isti pad energije. Prikaz omogućuje:

1. mijenjanje duljine i promjera grane 1;
2. mijenjanje duljine i promjera grane 2;
3. mijenjanje ukupnog protoka $Q$;
4. praćenje raspodjele $Q_1, Q_2$ i pripadnog gubitka energije.

## Pretpostavke modela

- razvijeno turbulentno strujanje u obje grane;
- konstantni koeficijent trenja $\lambda = 0{,}025$ za obje grane (uprošćeno);
- bez lokalnih gubitaka osim onih koje uključuje $\lambda$;
- voda kao radni fluid;
- grane povezuju iste ulazne i izlazne čvorove.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## Računski model

Gubitak energije u svakoj grani:

$$h_{w,i} = \lambda\,\frac{L_i}{D_i}\,\frac{v_i^2}{2g} = k_i\,Q_i^2, \quad k_i = \frac{8\lambda L_i}{\pi^2 g D_i^5}.$$

Uvjet paralelnog spoja: jednak pad energije u obje grane:

$$k_1 Q_1^2 = k_2 Q_2^2 \quad \Rightarrow \quad \frac{Q_1}{Q_2} = \sqrt{\frac{k_2}{k_1}}.$$

Uz uvjet $Q_1 + Q_2 = Q$, raspodjela se može zatvoriti analitički.

In [ ]:
G = 9.81
LAMBDA = 0.025

def k_grane(L, D_mm):
    D = D_mm / 1000.0
    return 8 * LAMBDA * L / (np.pi**2 * G * D**5)

def paralelne(L1, D1_mm, L2, D2_mm, Q_Lpsek):
    Q = Q_Lpsek / 1000.0  # m^3/s
    k1 = k_grane(L1, D1_mm)
    k2 = k_grane(L2, D2_mm)
    # Q1/Q2 = sqrt(k2/k1), Q1+Q2=Q
    omjer = np.sqrt(k2 / k1)
    Q2 = Q / (1 + omjer)
    Q1 = Q - Q2
    h_w = k1 * Q1**2
    return {'Q1': Q1, 'Q2': Q2, 'h_w': h_w,
             'udio1': Q1/Q, 'udio2': Q2/Q}

## Interaktivni prikaz

Klizačima u nastavku biraju se duljine i promjeri obje grane te ukupni protok. Prikaz pokazuje shemu paralelnog spoja s pripadnom raspodjelom protoka.

In [ ]:
def paralelne_prikaz(L1, D1_mm, L2, D2_mm, Q_Lpsek):
    r = paralelne(L1, D1_mm, L2, D2_mm, Q_Lpsek)

    fig, ax = plt.subplots(figsize=(10, 5.5))

    # Ulazni čvor (lijevo) i izlazni (desno)
    x_lijevo, x_desno = 0, 8
    y_grana1, y_grana2 = 1.5, -1.5

    # Ulazna i izlazna cijev
    ax.plot([-1.5, x_lijevo], [0, 0], color='#1565c0', lw=3)
    ax.plot([x_desno, x_desno + 1.5], [0, 0], color='#1565c0', lw=3)

    # Grane (debljina linije ovisi o D)
    debljina1 = max(1.5, D1_mm / 30)
    debljina2 = max(1.5, D2_mm / 30)
    ax.plot([x_lijevo, x_lijevo, x_desno, x_desno],
             [0, y_grana1, y_grana1, 0],
             color='#2e7d32', lw=debljina1, label='grana 1')
    ax.plot([x_lijevo, x_lijevo, x_desno, x_desno],
             [0, y_grana2, y_grana2, 0],
             color='#c62828', lw=debljina2, label='grana 2')

    # Oznake protoka i geometrije
    ax.text(x_desno/2, y_grana1 + 0.4,
             f'$L_1$ = {L1:.0f} m,  $D_1$ = {D1_mm:.0f} mm\n'
             f'$Q_1$ = {r["Q1"]*1000:.2f} L/s  ({r["udio1"]*100:.0f}%)',
             ha='center', color='#2e7d32', fontsize=10)
    ax.text(x_desno/2, y_grana2 - 0.4,
             f'$L_2$ = {L2:.0f} m,  $D_2$ = {D2_mm:.0f} mm\n'
             f'$Q_2$ = {r["Q2"]*1000:.2f} L/s  ({r["udio2"]*100:.0f}%)',
             ha='center', color='#c62828', fontsize=10, va='top')

    # Strelice protoka
    ax.annotate('', xy=(-0.5, 0), xytext=(-1.2, 0),
                 arrowprops=dict(arrowstyle='->',
                                   color='#1565c0', lw=2))
    ax.text(-1.4, 0.3, f'$Q$ = {Q_Lpsek:.1f} L/s',
             color='#1565c0', fontsize=10)

    ax.set_xlim(-2, x_desno + 2)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.set_title(f'Pad energije u obje grane:  '
                  f'$h_w$ = {r["h_w"]:.3f} m')
    ax.axis('off')

    plt.tight_layout()
    plt.show()


interact(
    paralelne_prikaz,
    L1=FloatSlider(min=10, max=200, step=5, value=80,
                    description='$L_1$ (m)',
                    layout=Layout(width='420px')),
    D1_mm=FloatSlider(min=20, max=200, step=5, value=80,
                       description='$D_1$ (mm)',
                       layout=Layout(width='420px')),
    L2=FloatSlider(min=10, max=200, step=5, value=80,
                    description='$L_2$ (m)',
                    layout=Layout(width='420px')),
    D2_mm=FloatSlider(min=20, max=200, step=5, value=120,
                       description='$D_2$ (mm)',
                       layout=Layout(width='420px')),
    Q_Lpsek=FloatSlider(min=1, max=30, step=0.5, value=10,
                         description='$Q$ (L/s)',
                         layout=Layout(width='420px'))
);

## Pitanja za istraživanje

1. **Identične grane.** Što se događa s raspodjelom protoka kada su obje grane potpuno jednake ($L_1 = L_2$, $D_1 = D_2$)? Zašto je rezultat 50:50 neovisno o samom protoku?

2. **Utjecaj promjera.** Pri $L_1 = L_2$ ali $D_2 = 2 D_1$, kako se raspoređuje protok? Koja je eksponentna ovisnost $Q_1/Q_2$ o omjeru promjera?

3. **Utjecaj duljine.** Pri istim promjerima i $L_2 = 4 L_1$, kako se raspoređuje protok? Zašto duljina ima manju polugu od promjera?

4. **Treća grana.** Ako se na ovaj sustav doda i treća paralelna grana, što se događa s ukupnim padom energije pri istom $Q$? Zašto se paralelni spoj uspoređuje s otpornicima u električnom krugu?

## Veza s teorijom poglavlja

Ovaj prikaz materijalizira temeljni princip paralelnog cjevovodnog spoja iz poglavlja U13: protok se raspoređuje tako da svaka grana ima isti pad ukupne energije između zajedničkih čvorova. Šira ili kraća grana spontano preuzima veći dio protoka — ne zato što joj je propisan, nego zato što na istom dopuštenom padu energije može propustiti više tekućine. Princip se proširuje na cijele cjevovodne mreže gdje se sustav rješava iterativno (Hardy-Crossova metoda).